# 3) Multi-period dispatch — Exercises

**Goals:** check ramp feasibility and simulate a battery by hand, build your own 24-hour Linopy model with a flexible coal ramp and battery size and match PyPSA, put a price on flexibility, and find where extra storage stops paying.

The setup cell holds the lecture's day (`LOAD`, `FREE`, `PLANTS`, `ETA`) and a PyPSA helper for checking. Power in MW, energy in MWh (one-hour steps), prices in €/MWh, cost in € per day.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import logging
import warnings

import linopy
import numpy as np
import pandas as pd
import pypsa
import xarray as xr

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

# the lecture's day
T = 24
HOURS = np.arange(T)
LOAD = 1900 + 400 * np.sin(np.pi * (HOURS - 6) / 12).clip(0) + 250 * np.exp(-((HOURS - 19) / 2.0) ** 2)
FREE = 2200 * 0.8 * np.clip(np.sin(np.pi * (HOURS - 6) / 13), 0, None) + 400.0
PLANTS = pd.DataFrame({"p_nom": [900, 900, 500], "mc": [60, 95, 150], "ramp": [60, 300, 500]},
                      index=pd.Index(["coal", "ccgt", "ocgt"], name="gen"))
ETA = 0.95


def pypsa_day(coal_ramp=60.0, battery_mw=300.0, battery_mwh=1200.0):
    """The lecture's day in PyPSA, for checking. Returns (cost EUR, hourly prices EUR/MWh)."""
    n = pypsa.Network()
    n.set_snapshots(HOURS)
    n.add("Bus", "HR")
    n.add("Load", "load", bus="HR", p_set=LOAD)
    n.add("Generator", "free", bus="HR", p_nom=FREE.max(), p_max_pu=FREE / FREE.max(), marginal_cost=0)
    for gen, row in PLANTS.iterrows():
        ramp = coal_ramp if gen == "coal" else row.ramp
        n.add("Generator", gen, bus="HR", p_nom=row.p_nom, marginal_cost=row.mc,
              ramp_limit_up=ramp / row.p_nom, ramp_limit_down=ramp / row.p_nom)
    if battery_mwh > 0:
        n.add("StorageUnit", "battery", bus="HR", p_nom=battery_mw, max_hours=battery_mwh / battery_mw,
              efficiency_store=ETA, efficiency_dispatch=ETA, cyclic_state_of_charge=True)
    n.optimize(solver_name="highs", log_to_console=False, progress=False)
    return n.objective, n.buses_t.marginal_price["HR"].to_numpy()

### Warm-ups

**1. Is a schedule ramp-feasible?**

Write `ramp_violation(profile, ramp)` returning the largest amount (MW) by which consecutive hours of a schedule exceed the ramp limit (0 if it is feasible).

In [ ]:
def ramp_violation(profile, ramp: float) -> float:
    """Largest excess of |g[t] - g[t-1]| over the ramp limit (MW); 0 if feasible."""
    ...  # YOUR CODE HERE

In [ ]:
assert ramp_violation([900, 840, 780, 840], 60) == 0
assert np.isclose(ramp_violation([900, 900, 600], 60), 240)
assert ramp_violation([500], 10) == 0

**2. Simulate a battery**

Write `soc_path(charge, discharge, e_start, eta=ETA)` returning the state of charge after each hour (MWh): e[t] = e[t−1] + η·c[t] − d[t]/η.

In [ ]:
def soc_path(charge, discharge, e_start: float, eta: float = ETA) -> np.ndarray:
    """State of charge (MWh) at the end of each hour."""
    ...  # YOUR CODE HERE

In [ ]:
path = soc_path([100, 100, 0, 0], [0, 0, 90.25, 90.25], e_start=0)
assert np.allclose(path, [95, 190, 95, 0]), "charge 200 MWh, get back 0.95 x 0.95 x 200 = 180.5 MWh"
assert np.isclose(soc_path([0], [95], e_start=100)[-1], 0), "discharging 95 MWh empties 100 MWh of storage"

### Core

**3. Your own model, with knobs**

Write `dispatch(coal_ramp, battery_mw=300, battery_mwh=1200)` — the lecture's Linopy model with the coal ramp limit and the battery size as parameters (`battery_mwh=0` means no battery). Return a dict with `cost`, `prices` (array) and `curtailed` (MWh). Check it against `pypsa_day` for three different settings.

In [ ]:
def dispatch(coal_ramp: float, battery_mw: float = 300.0, battery_mwh: float = 1200.0) -> dict:
    """24-hour dispatch with ramp limits and an optional cyclic battery."""
    ...  # YOUR CODE HERE

In [ ]:
for settings in [dict(coal_ramp=60), dict(coal_ramp=150, battery_mwh=0), dict(coal_ramp=30, battery_mw=200, battery_mwh=400)]:
    mine = dispatch(**settings)
    cost_ref, prices_ref = pypsa_day(**settings)
    assert np.isclose(mine["cost"], cost_ref, rtol=1e-9), settings
    assert np.abs(mine["prices"] - prices_ref).max() < 1e-6, settings
assert np.isclose(dispatch(60)["cost"], 1_837_742.65, atol=0.01), "the lecture's ramp + battery case"
print("own model = PyPSA (cost and every hourly price) for all three settings")

**4. What is a flexible coal plant worth?**

Without a battery, sweep the coal ramp limit over 20, 40, 60, 100, 150, 200 and 300 MW/h. Write `ramp_sweep(ramps)` returning a DataFrame (index: ramp) with daily cost, curtailment and maximum price. Find the smallest ramp in your sweep at which no solar is curtailed.

In [ ]:
def ramp_sweep(ramps) -> pd.DataFrame:
    """Cost (EUR/day), curtailment (MWh) and maximum price (EUR/MWh) for each coal ramp limit, without battery."""
    ...  # YOUR CODE HERE

In [ ]:
sweep = ramp_sweep([20, 40, 60, 100, 150, 200, 300])
assert sweep.cost.is_monotonic_decreasing, "a more flexible plant never makes the day more expensive"
no_curtailment = sweep.index[sweep.curtailed < 1e-6].min()
assert sweep.loc[20, "curtailed"] > sweep.loc[60, "curtailed"] > 0
print(f"no curtailment from a ramp of {no_curtailment} MW/h; going from 20 to 300 MW/h saves "
      f"{sweep.loc[20, 'cost'] - sweep.loc[300, 'cost']:,.0f} EUR/day")
sweep.round(1)

**5. Interpretation**

Your sweep puts a price on flexibility. Why does the saving per extra MW/h of ramp rate shrink as the plant gets more flexible? And why do inflexible plants make solar curtailment — and zero or negative prices — more likely, even when total supply would be enough?

*Your answer here.*

### Challenge

**6. How much storage is worth building?**

With the 60 MW/h coal ramp and a 300 MW battery, sweep the battery's energy capacity over 0, 150, 300, 600, 1200, 1800, 2400 and 3600 MWh. Write `storage_sweep(sizes)` returning, for each size, the daily cost, the saving relative to no battery, and the **marginal** saving of the last step per MWh of added capacity. Confirm one size with PyPSA.

In [ ]:
def storage_sweep(sizes) -> pd.DataFrame:
    """Daily cost, total saving vs no battery, and marginal saving per added MWh, for each battery energy size."""
    ...  # YOUR CODE HERE

In [ ]:
storage = storage_sweep([0, 150, 300, 600, 1200, 1800, 2400, 3600])
marginal = storage.marginal_saving_per_MWh.dropna()
assert (storage.cost.diff().dropna() <= 1e-6).all(), "more storage never raises cost"
assert marginal.is_monotonic_decreasing, "each extra MWh of storage is worth less than the one before"
assert np.isclose(marginal.iloc[-1], 0, atol=1e-6), "beyond some size, extra storage energy is worth nothing"
cost_ref, _ = pypsa_day(60, 300, 600)
assert np.isclose(storage.loc[600, "cost"], cost_ref, rtol=1e-9)
storage.round(1)

**7. Interpretation**

Where does the value of extra storage energy run out in this day, and why? Explain the plateau at 90.25 €/MWh. A developer argues "the first 150 MWh earned X € per MWh per day, so a 2,400 MWh battery will earn 16 times as much". What is wrong with that, and what would you need to look at beyond this single day before deciding on a size?

*Your answer here.*